# Crux — Fine-tuning a YOLO26 Hold Detector (Transfer Learning)

**The graded core.** COCO — what `yolo26n.pt` is pretrained on — has **no `hold` class**.
So a COCO model scores ~0 mAP on climbing holds. We **fine-tune** it on a single-class
`hold` dataset (~417 images, 640×640) and measure the **before → after mAP gain**. That
gain *is* the transfer-learning result this notebook is graded on.

Where it fits: detect holds **once** on an empty-wall frame → a static hold map; a
pretrained pose model runs per frame; fusing them yields a climb **debrief** (contacts,
crux, feet cuts). This notebook owns the detector + its evaluation.

> Run top-to-bottom on **Colab (GPU)**. All logic lives in `training/train.py` and
> `training/eval.py` — the cells orchestrate and narrate, nothing is buried here.

## 1. Setup

> **Run this against a checkout that has the Model-role code** (`training/train.py` +
> `training/eval.py` with `eval.run()` returning the before/after summary dict). That means
> the `jan/model` branch **merged to `main`**, or the clone below pinned to the branch
> (`git clone -b jan/model ...`). A plain clone of `main` before the merge gets the old stubs
> and the eval/notebook cells will not match.

In [ ]:
import os, sys
# Colab: clone the repo once. Local: this is a no-op (config.yaml already present).
if not os.path.exists("config.yaml") and not os.path.exists("Crux"):
    !git clone -q https://github.com/moanv2/Crux
if os.path.exists("Crux"):
    %cd Crux
!pip install -q -r requirements.txt

In [ ]:
sys.path.insert(0, "data/scripts")
sys.path.insert(0, "training")
from config import load_config, resolve_paths
import train
import eval as evaluate          # Model-role modules — all logic lives here, not in cells
cfg = load_config()
paths = resolve_paths(cfg)
print("image_size:", cfg["image_size"], "| classes:", cfg["classes"])

## 2. Get the dataset (Diego's pipeline)

Pulls Roboflow v3 → cleans → splits → writes `data/data.yaml` (single class `hold`).

In [ ]:
os.environ.setdefault("ROBOFLOW_API_KEY", "")   # paste your key, or set it as a Colab secret
assert os.environ["ROBOFLOW_API_KEY"], "Set ROBOFLOW_API_KEY (Roboflow -> Settings -> API)."
!python data/scripts/run_pipeline.py
from pathlib import Path
print("data.yaml exists:", paths.data_yaml.exists())
print(Path("data/data_card.md").read_text())

## 3. Peek at the data

In [ ]:
import yaml, matplotlib.pyplot as plt
from PIL import Image
d = yaml.safe_load(paths.data_yaml.read_text())
imgs = sorted((Path(d["path"]) / "train" / "images").glob("*"))[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.ravel(), imgs):
    ax.imshow(Image.open(p)); ax.set_title(p.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Baseline (before): COCO-pretrained, zero-shot

Expect **~0 mAP** — COCO has no `hold` class, so the pretrained model cannot find holds. This is the *before* in the before/after.

In [ ]:
from ultralytics import YOLO
base = YOLO(cfg["training"]["model"])    # COCO-pretrained — no `hold` class
split = evaluate.report_split(str(paths.data_yaml))   # held-out test split (fallback: val)
base_m = base.val(data=str(paths.data_yaml), imgsz=cfg["image_size"], split=split, name="nb_baseline")
print(f"Baseline ({split})  mAP50={base_m.box.map50:.4f}  mAP50-95={base_m.box.map:.4f}")

## 5. Fine-tune (transfer learning)

Reads every knob (epochs, batch, lr, augmentation) from `config.yaml`; copies the best checkpoint to `models/best.pt`. **Long GPU cell.**

In [ ]:
weights = train.run(cfg, paths)   # -> models/best.pt
print("Frozen weights:", weights)

## 6. Evaluate (after) + handoff exports

Vals `best.pt` and the COCO baseline on the same data, writes `training/artifacts/metrics.{json,md}`, and copies the PR curve / confusion matrix / sample predictions there for Dalton.

In [ ]:
summary = evaluate.run(cfg, paths)
from IPython.display import Markdown, display
display(Markdown((paths.root / "training" / "artifacts" / "metrics.md").read_text()))

## 7. Before / after — the transfer-learning gain

The money shot: COCO baseline vs fine-tuned, side by side.

In [ ]:
import matplotlib.pyplot as plt
labels = ["mAP50", "mAP50-95"]
base_v = [summary["baseline"]["map50"], summary["baseline"]["map"]]
fine_v = [summary["finetuned"]["map50"], summary["finetuned"]["map"]]
x = range(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([i - w/2 for i in x], base_v, w, label="COCO baseline")
ax.bar([i + w/2 for i in x], fine_v, w, label="Fine-tuned")
ax.set_xticks(list(x)); ax.set_xticklabels(labels); ax.set_ylim(0, 1)
ax.set_title("Transfer learning: COCO -> fine-tuned hold detector"); ax.legend()
for i, (b, f) in enumerate(zip(base_v, fine_v)):
    ax.text(i - w/2, b + 0.01, f"{b:.2f}", ha="center")
    ax.text(i + w/2, f + 0.01, f"{f:.2f}", ha="center")
plt.tight_layout()
plt.savefig(paths.root / "training" / "artifacts" / "before_after.png", dpi=150); plt.show()

## 8. Qualitative: PR curve + sample detections

In [ ]:
from IPython.display import Image as IPImage, display
art = paths.root / "training" / "artifacts"
for fig in ["BoxPR_curve.png", "PR_curve.png", "confusion_matrix.png"]:
    if (art / fig).exists():
        display(IPImage(str(art / fig)))

## 9. Inference demo — static hold map (-> Ignacio)

Run `best.pt` once on an empty-wall reference frame. This is exactly the hold map Ignacio's pipeline consumes.

In [ ]:
split = evaluate.report_split(str(paths.data_yaml))
imgs = sorted(p for p in (Path(d["path"]) / split / "images").glob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if not imgs:
    print("No reference image found — skipping inference demo.")
else:
    ref = imgs[0]
    det = YOLO(str(paths.weights)).predict(source=str(ref), imgsz=cfg["image_size"], save=True)
    print(f"Detected {len(det[0].boxes)} holds on {ref.name}")

## 10. Handoff
- **`models/best.pt`** -> Ignacio (Integration): the one frozen detector.
- **`training/artifacts/`** (`metrics.json`, `metrics.md`, `before_after.png`, PR curve,
  confusion matrix, `samples/`) -> Dalton (Results/Docs) for the slides.